In [ ]:
import sys
sys.path.insert(0, '..')

import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

from ecg_ssl import config
from ecg_ssl.dataset import build_dataloaders
from ecg_ssl.model import ECGMaskedSSL
from ecg_ssl.trainer import run_epoch

print('Device:', config.device)

In [ ]:
assert config.BASE.exists(), f'Base path does not exist: {config.BASE}'

hea_files = sorted(config.BASE.rglob('*.hea'))
record_paths = [str(p.with_suffix('')) for p in hea_files]
record_paths = [
    rp for rp in record_paths
    if Path(rp + '.hea').exists() and Path(rp + '.dat').exists()
]
print(f'Usable file-paired records found: {len(record_paths):,}')

if config.MAX_RECORDS is not None:
    record_paths = record_paths[:config.MAX_RECORDS]
    print(f'Using first {len(record_paths):,} records')
else:
    print(f'Using all {len(record_paths):,} records')

train_loader, val_loader = build_dataloaders(record_paths)
print(f'Train size: {len(train_loader.dataset):,}')
print(f'Val size  : {len(val_loader.dataset):,}')

In [ ]:
CHECKPOINT_DIR = Path('../checkpoints_fixed_ssl_rawpatch')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

model = ECGMaskedSSL(
    in_channels=config.IN_CHANNELS,
    seq_len=config.SEQ_LEN,
    d_model=config.D_MODEL,
    patch_size=config.PATCH_SIZE,
    num_heads=config.NUM_HEADS,
    num_layers=config.NUM_LAYERS,
    mlp_ratio=config.MLP_RATIO,
    dropout=config.DROPOUT,
).to(config.device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LR,
    weight_decay=config.WEIGHT_DECAY,
)

total_steps = config.EPOCHS * len(train_loader)
warmup_steps = int(0.05 * total_steps)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step + 1) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'Total train steps : {total_steps:,}')
print(f'Warmup steps      : {warmup_steps:,}')

In [ ]:
best_val_loss = float('inf')
history = []
step_history = {'step': [], 'batch_loss': [], 'ema_loss': []}
global_step = 0

for epoch in range(1, config.EPOCHS + 1):
    print(f'\nEpoch {epoch}/{config.EPOCHS}')

    train_loss, global_step = run_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        train=True,
        global_step=global_step,
        plot_every=config.PLOT_EVERY,
        step_history=step_history,
    )

    val_loss, _ = run_epoch(
        model=model,
        loader=val_loader,
        train=False,
        global_step=global_step,
    )

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'config': {
            'PATCH_SIZE': config.PATCH_SIZE,
            'D_MODEL': config.D_MODEL,
            'NUM_HEADS': config.NUM_HEADS,
            'NUM_LAYERS': config.NUM_LAYERS,
            'MASK_RATIO': config.MASK_RATIO,
            'MASK_SPAN_LEN': config.MASK_SPAN_LEN,
            'BATCH_SIZE': config.BATCH_SIZE,
            'LR': config.LR,
            'objective': 'raw_patch_reconstruction',
        },
    }
    torch.save(checkpoint, CHECKPOINT_DIR / 'latest.pt')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, CHECKPOINT_DIR / 'best.pt')
        print(f'  Saved new best checkpoint to: {CHECKPOINT_DIR / "best.pt"}')

print('\nDone training.')
print('Best val loss:', best_val_loss)
print('Checkpoint dir:', CHECKPOINT_DIR)

epochs_list = [x['epoch'] for x in history]
train_losses = [x['train_loss'] for x in history]
val_losses = [x['val_loss'] for x in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_list, train_losses, marker='o', label='Train loss')
plt.plot(epochs_list, val_losses, marker='o', label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Epoch-Level Raw Patch Reconstruction Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# # =========================
# # Inspect corrupted / flat / repaired records
# # =========================
# from collections import Counter
# import numpy as np
# import wfdb

# N_INSPECT = 799903  # inspect all records

# def inspect_record_quality(record_paths, n_inspect=20000, flat_std_thresh=1e-4):
#     stats = Counter()
#     examples = {
#         'bad_shape': [],
#         'all_nan_lead': [],
#         'nan_or_inf_repaired': [],
#         'flat_lead': [],
#         'nonfinite_after_repair': [],
#         'read_error': [],
#     }

#     paths = record_paths[:min(n_inspect, len(record_paths))]

#     for i, rp in enumerate(paths, 1):
#         try:
#             rec = wfdb.rdrecord(rp)
#             x = rec.p_signal.astype(np.float32).T   # (C, T)
#         except Exception as e:
#             stats['read_error'] += 1
#             if len(examples['read_error']) < 5:
#                 examples['read_error'].append((rp, repr(e)))
#             continue

#         stats['total'] += 1

#         # shape check
#         if x.shape != (12, 5000):
#             stats['bad_shape'] += 1
#             if len(examples['bad_shape']) < 5:
#                 examples['bad_shape'].append((rp, x.shape))
#             continue

#         had_nan_or_inf = np.isnan(x).any() or np.isinf(x).any()

#         # replace inf -> nan
#         x = np.where(np.isinf(x), np.nan, x)

#         # all-NaN lead
#         if np.isnan(x).all(axis=1).any():
#             stats['all_nan_lead'] += 1
#             if len(examples['all_nan_lead']) < 5:
#                 bad_leads = np.where(np.isnan(x).all(axis=1))[0].tolist()
#                 examples['all_nan_lead'].append((rp, bad_leads))
#             continue

#         # repair partial NaNs with per-lead median
#         repaired = False
#         for c in range(x.shape[0]):
#             lead = x[c]
#             if np.isnan(lead).any():
#                 med = np.nanmedian(lead)
#                 if np.isnan(med):
#                     stats['all_nan_lead'] += 1
#                     if len(examples['all_nan_lead']) < 5:
#                         examples['all_nan_lead'].append((rp, [c]))
#                     repaired = None
#                     break
#                 x[c] = np.where(np.isnan(lead), med, lead)
#                 repaired = True

#         if repaired is None:
#             continue

#         if had_nan_or_inf or repaired:
#             stats['nan_or_inf_repaired'] += 1
#             if len(examples['nan_or_inf_repaired']) < 5:
#                 examples['nan_or_inf_repaired'].append(rp)

#         if not np.isfinite(x).all():
#             stats['nonfinite_after_repair'] += 1
#             if len(examples['nonfinite_after_repair']) < 5:
#                 examples['nonfinite_after_repair'].append(rp)
#             continue

#         # flat lead check before normalization
#         std = x.std(axis=1)
#         if (std < flat_std_thresh).any():
#             stats['flat_lead'] += 1
#             if len(examples['flat_lead']) < 5:
#                 bad_leads = np.where(std < flat_std_thresh)[0].tolist()
#                 examples['flat_lead'].append((rp, bad_leads, std.tolist()))
#             continue

#         stats['usable'] += 1

#     return stats, examples

# stats, examples = inspect_record_quality(record_paths, n_inspect=N_INSPECT)

# print(f'Inspected: {stats["total"]}')
# for k in [
#     'usable',
#     'bad_shape',
#     'read_error',
#     'all_nan_lead',
#     'nan_or_inf_repaired',
#     'nonfinite_after_repair',
#     'flat_lead',
# ]:
#     print(f'{k:22s}: {stats[k]}')

# if stats['total'] > 0:
#     print('\nPercentages over inspected readable-shape attempts:')
#     for k in ['usable', 'all_nan_lead', 'nan_or_inf_repaired', 'flat_lead']:
#         print(f'{k:22s}: {100 * stats[k] / stats["total"]:.4f}%')

# print('\nExample problematic records:')
# for k, vals in examples.items():
#     if vals:
#         print(f'\n{k}:')
#         for v in vals:
#             print('  ', v)